# 🏏 IPL 2026 — Analytics Dashboard Generator
**Scrapes live stats from the CSV and auto-regenerates the HTML. Re-run any cell after updating the CSV.**

## Cell 1 — Install & Import

In [ ]:
# ─── INSTALL (run once) ───────────────────────────────────────────────────────
# !pip install pandas jinja2

import pandas as pd
import json, os, warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

## Cell 2 — Load CSV

In [ ]:
# ─── LOAD CSV ────────────────────────────────────────────────────────────────
CSV_PATH = 'ipl_data.csv'   # <-- change this to your CSV filename
TARGET_SEASON = '2026'      # <-- change to any season e.g. '2025', '2024'

df = pd.read_csv(CSV_PATH, low_memory=False)
df26 = df[df['season'].astype(str) == TARGET_SEASON].copy()

print(f'✅ Loaded {len(df):,} total rows')
print(f'✅ {TARGET_SEASON} season: {len(df26):,} ball-by-ball rows')
print(f'   Columns: {list(df26.columns)}')

## Cell 3 — Compute Hero Stats

In [ ]:
# ─── HERO STATS ──────────────────────────────────────────────────────────────
matches_df = df26.drop_duplicates('match_id')[[
    'match_id','date','toss_winner','toss_decision',
    'winner','win_by_runs','win_by_wickets','team1','team2','player_of_match'
]].copy()

total_matches = len(matches_df)
total_teams   = df26[['team1','team2']].stack().nunique()

# Top run scorer
top_runs_series = df26.groupby('batter')['runs_batter'].sum().sort_values(ascending=False)
top_bat_name    = top_runs_series.index[0]
top_bat_runs    = int(top_runs_series.iloc[0])

# Top wicket taker (exclude non-bowling dismissals)
non_bowl_kinds = ['run out', 'retired hurt', 'obstructing the field']
wkt_df  = df26[df26['wicket_player_out'].notna() & ~df26['wicket_kind'].isin(non_bowl_kinds)]
top_wkt_series = wkt_df.groupby('bowler').size().sort_values(ascending=False)
top_bowl_name  = top_wkt_series.index[0]
top_bowl_wkts  = int(top_wkt_series.iloc[0])

hero = {
    'total_matches': total_matches,
    'total_teams':   total_teams,
    'top_bat_name':  top_bat_name,
    'top_bat_runs':  top_bat_runs,
    'top_bowl_name': top_bowl_name,
    'top_bowl_wkts': top_bowl_wkts,
}

print('✅ Hero stats:')
for k, v in hero.items():
    print(f'   {k}: {v}')

## Cell 4 — Toss Analysis

In [ ]:
# ─── TOSS ANALYSIS ───────────────────────────────────────────────────────────
n = len(matches_df)

toss_wins_match = (matches_df['toss_winner'] == matches_df['winner']).sum()
toss_win_pct    = round(toss_wins_match / n * 100, 1)
toss_lose_pct   = round(100 - toss_win_pct, 1)

field_df  = matches_df[matches_df['toss_decision'] == 'field']
bat_df    = matches_df[matches_df['toss_decision'] == 'bat']

field_n         = len(field_df)
bat_n           = len(bat_df)
field_pct       = round(field_n / n * 100, 1)
field_win_pct   = round((field_df['toss_winner'] == field_df['winner']).sum() / field_n * 100, 1) if field_n else 0
bat_win_pct     = round((bat_df['toss_winner'] == bat_df['winner']).sum() / bat_n * 100, 1) if bat_n else 0
margin_pct      = round(abs(toss_win_pct - 50), 1)

toss = {
    'toss_win_pct':   toss_win_pct,
    'toss_lose_pct':  toss_lose_pct,
    'toss_wins':      int(toss_wins_match),
    'toss_losses':    int(n - toss_wins_match),
    'total_matches':  n,
    'field_pct':      field_pct,
    'field_win_pct':  field_win_pct,
    'bat_pct':        round(100 - field_pct, 1),
    'bat_win_pct':    bat_win_pct,
    'margin_pct':     margin_pct,
}

print('✅ Toss analysis:')
for k, v in toss.items():
    print(f'   {k}: {v}')

## Cell 5 — Phase Analysis

In [ ]:
# ─── PHASE ANALYSIS (WINNER vs LOSER) ────────────────────────────────────────
df26['phase'] = pd.cut(
    df26['over'],
    bins=[-1, 5, 14, 19],
    labels=['Powerplay', 'Middle', 'Death']
)

phases_data = []
phase_meta  = {
    'Powerplay': 'Overs 1–6',
    'Middle':    'Overs 7–15',
    'Death':     'Overs 16–20',
}
match_winner_map = matches_df.set_index('match_id')['winner'].to_dict()
match_bat1_map   = df26[df26['innings']==1].drop_duplicates('match_id').set_index('match_id')['batting_team'].to_dict()

for phase in ['Powerplay', 'Middle', 'Death']:
    pdata = df26[(df26['phase']==phase) & (df26['innings']==1)].copy()
    per_match = pdata.groupby('match_id')['runs_total'].sum().reset_index()
    per_match['batting_team'] = per_match['match_id'].map(match_bat1_map)
    per_match['winner']       = per_match['match_id'].map(match_winner_map)
    per_match['is_winner']    = per_match['batting_team'] == per_match['winner']
    avg = per_match.groupby('is_winner')['runs_total'].mean()
    w_avg = round(float(avg.get(True,  0)), 1)
    l_avg = round(float(avg.get(False, 0)), 1)
    gap   = round(w_avg - l_avg, 1)
    max_r = max(w_avg, l_avg, 1)
    phases_data.append({
        'name':    phase,
        'overs':   phase_meta[phase],
        'w_avg':   w_avg,
        'l_avg':   l_avg,
        'gap':     gap,
        'w_pct':   round(w_avg / max_r * 100, 1),
        'l_pct':   round(l_avg / max_r * 100, 1),
        'highlight': phase == max(phases_data + [{'gap': gap, 'name': phase}], key=lambda x: x['gap'])['name'] if phases_data else False
    })

# Mark the phase with biggest gap
biggest_gap_phase = max(phases_data, key=lambda x: x['gap'])['name']
for p in phases_data:
    p['highlight'] = (p['name'] == biggest_gap_phase)

print('✅ Phase analysis:')
for p in phases_data:
    print(f"   {p['name']}: Winner {p['w_avg']} vs Loser {p['l_avg']} (gap +{p['gap']}) {'⭐ BIGGEST GAP' if p['highlight'] else ''}")

## Cell 6 — Player Rankings

In [ ]:
# ─── PLAYER RANKINGS ─────────────────────────────────────────────────────────
batter_team_map = (
    df26[['batter','batting_team']]
    .drop_duplicates('batter')
    .set_index('batter')['batting_team']
    .to_dict()
)

# Top 5 batters
bat_runs   = df26.groupby('batter')['runs_batter'].sum().sort_values(ascending=False).head(5)
bat_matches = df26.groupby('batter')['match_id'].nunique()
bat_sr      = (df26.groupby('batter')['runs_batter'].sum() /
               df26.groupby('batter').size() * 100).round(1)

top_batters = []
for i, (name, runs) in enumerate(bat_runs.items()):
    team = batter_team_map.get(name, '???')
    # abbreviate team name
    abbr = {'Sunrisers Hyderabad':'SRH','Royal Challengers Bengaluru':'RCB',
            'Punjab Kings':'PBKS','Rajasthan Royals':'RR','Gujarat Titans':'GT',
            'Chennai Super Kings':'CSK','Delhi Capitals':'DC',
            'Kolkata Knight Riders':'KKR','Mumbai Indians':'MI',
            'Lucknow Super Giants':'LSG'}.get(team, team[:3].upper())
    top_batters.append({
        'rank': i+1, 'name': name, 'team': abbr,
        'runs': int(runs), 'sr': float(bat_sr.get(name, 0)),
        'matches': int(bat_matches.get(name, 0))
    })

# Top 5 bowlers
top_wkt_series = wkt_df.groupby('bowler').size().sort_values(ascending=False).head(5)

# Map bowler to their team via non-batting innings
bowler_team_raw = (
    df26[df26['innings']==2][['bowler','batting_team']]
    .drop_duplicates('bowler')
)
# Bowler appears in innings where their TEAM is NOT batting
# Use innings 1 ball data -> batting_team is the batting team; bowler is from FIELDING team
# We'll just skip team for bowlers or use a basic lookup
bowl_eco = {}
bowl_matches = df26.groupby('bowler')['match_id'].nunique()
for b in top_wkt_series.index:
    runs = df26[df26['bowler']==b]['runs_total'].sum()
    balls = df26[df26['bowler']==b]['ball'].count()
    bowl_eco[b] = round(runs / (balls/6), 2) if balls else 0

top_bowlers = []
for i, (name, wkts) in enumerate(top_wkt_series.items()):
    top_bowlers.append({
        'rank': i+1, 'name': name,
        'wkts': int(wkts), 'eco': bowl_eco.get(name, 0),
        'matches': int(bowl_matches.get(name, 0))
    })

print('✅ Top Batters:')
for b in top_batters:
    print(f"   #{b['rank']} {b['name']} ({b['team']}) — {b['runs']} runs | SR {b['sr']} | {b['matches']} matches")

print()
print('✅ Top Bowlers:')
for b in top_bowlers:
    print(f"   #{b['rank']} {b['name']} — {b['wkts']} wkts | Eco {b['eco']} | {b['matches']} matches")

## Cell 7 — Standings

In [ ]:
# ─── STANDINGS ───────────────────────────────────────────────────────────────
all_teams = sorted(set(matches_df['team1'].tolist() + matches_df['team2'].tolist()))

# NRR: (runs scored / overs faced) - (runs conceded / overs bowled)
def calc_nrr(team):
    t_bat  = df26[(df26['batting_team']==team)]
    t_bowl = df26[(df26['batting_team']!=team) & (
                  (df26['match_id'].isin(matches_df[matches_df['team1']==team]['match_id'])) |
                  (df26['match_id'].isin(matches_df[matches_df['team2']==team]['match_id']))
                 )]
    rs  = t_bat['runs_total'].sum()
    os  = t_bat['ball'].count() / 6
    rc  = t_bowl['runs_total'].sum()
    oc  = t_bowl['ball'].count() / 6
    if os == 0 or oc == 0:
        return 0.0
    return round((rs/os) - (rc/oc), 3)

standings_raw = []
for team in all_teams:
    m = matches_df[(matches_df['team1']==team) | (matches_df['team2']==team)]
    w = int((m['winner']==team).sum())
    p = len(m)
    l = p - w
    standings_raw.append({
        'team': team, 'W': w, 'L': l, 'P': p,
        'PTS': w*2, 'NRR': calc_nrr(team)
    })

standings = sorted(standings_raw, key=lambda x: (-x['PTS'], -x['NRR']))

# Qualify top 4
for i, t in enumerate(standings):
    t['pos'] = i+1
    t['status'] = 'qualified' if i < 4 else ('eliminated' if i >= 5 else '')

print('✅ Standings:')
for s in standings:
    print(f"   #{s['pos']} {s['team']}: {s['W']}W {s['L']}L | NRR {s['NRR']:+.3f} | {s['PTS']} pts  [{s['status']}]")

## Cell 8 — Surprise Stat

In [ ]:
# ─── SURPRISE STAT ───────────────────────────────────────────────────────────
# Youngest / breakout batter: lowest match count among top-10 run scorers
top10_bat = df26.groupby('batter')['runs_batter'].sum().sort_values(ascending=False).head(10)
bat_m10   = bat_matches[top10_bat.index]

# Death overs gap
death_phase = [p for p in phases_data if p['name']=='Death'][0]
death_gap   = death_phase['gap']

surprise = {
    'toss_win_pct':     toss_win_pct,
    'field_win_pct':    field_win_pct,
    'death_gap':        death_gap,
    'surprise_player':  top_bat_name,
    'surprise_runs':    top_bat_runs,
    'top_bowl_name':    top_bowl_name,
    'top_bowl_wkts':    top_bowl_wkts,
}

print('✅ Surprise section stats:')
for k, v in surprise.items():
    print(f'   {k}: {v}')

## Cell 9 — Generate HTML

In [ ]:
# ─── HTML TEMPLATE & GENERATOR ───────────────────────────────────────────────

def rank_class(r):
    return {1:'r1', 2:'r2', 3:'r3'}.get(r, 'rn')

def sign(v):
    return f'+{v}' if v >= 0 else str(v)

def render_batter_rows(batters):
    rows = []
    for b in batters:
        rows.append(f"""
        <div class="player-row">
          <div class="pr-rank {rank_class(b['rank'])}">{b['rank']}</div>
          <div class="pr-info">
            <div class="pr-name">{b['name']}</div>
            <div class="pr-meta">{b['team']} &middot; SR {b['sr']}</div>
          </div>
          <div class="pr-stats">
            <span class="pr-main bat">{b['runs']}</span>
            <div class="pr-sub">{b['matches']} matches</div>
          </div>
        </div>""")
    return '\n'.join(rows)

def render_bowler_rows(bowlers):
    rows = []
    for b in bowlers:
        rows.append(f"""
        <div class="player-row">
          <div class="pr-rank {rank_class(b['rank'])}">{b['rank']}</div>
          <div class="pr-info">
            <div class="pr-name">{b['name']}</div>
            <div class="pr-meta">Eco {b['eco']}</div>
          </div>
          <div class="pr-stats">
            <span class="pr-main bowl">{b['wkts']}</span>
            <div class="pr-sub">Eco {b['eco']} &middot; {b['matches']} matches</div>
          </div>
        </div>""")
    return '\n'.join(rows)

def render_phase_blocks(phases):
    blocks = []
    for p in phases:
        hl = 'highlight' if p['highlight'] else ''
        blocks.append(f"""
      <div class="phase-block {hl}">
        <div class="pb-left">
          <div class="pb-name">{p['name'].upper()}</div>
          <div class="pb-overs">{p['overs']}</div>
          <div class="pb-gap-badge">Gap +{p['gap']} runs</div>
        </div>
        <div class="pb-right">
          <div class="pb-bar-row">
            <div class="pb-bar-label"><span>Winner avg</span><span>{p['w_avg']}</span></div>
            <div class="pb-bar-track"><div class="pb-bar-fill winner" data-w="{p['w_pct']}">{p['w_avg']}</div></div>
            <div class="pb-bar-label"><span>Loser avg</span><span>{p['l_avg']}</span></div>
            <div class="pb-bar-track"><div class="pb-bar-fill loser" data-w="{p['l_pct']}">{p['l_avg']}</div></div>
          </div>
        </div>
      </div>""")
    return '\n'.join(blocks)

def render_standings_rows(standings):
    rows = []
    abbr_map = {
        'Sunrisers Hyderabad':'SRH','Royal Challengers Bengaluru':'RCB',
        'Punjab Kings':'PBKS','Rajasthan Royals':'RR','Gujarat Titans':'GT',
        'Chennai Super Kings':'CSK','Delhi Capitals':'DC',
        'Kolkata Knight Riders':'KKR','Mumbai Indians':'MI',
        'Lucknow Super Giants':'LSG'
    }
    for s in standings:
        cls  = s['status']
        code = abbr_map.get(s['team'], s['team'][:3].upper())
        nrr_str = sign(s['NRR'])
        w_col = 'style="color:var(--green)"' if s['W'] >= 5 else ''
        l_col = 'style="color:var(--red)"'   if s['L'] >= 5 else ''
        rows.append(f"""
      <div class="pt-row {cls}">
        <div class="pt-pos">{s['pos']}</div>
        <div><div class="pt-team">{s['team']}</div><div class="pt-team-code">{code}</div></div>
        <div class="pt-val" {w_col}>{s['W']}</div>
        <div class="pt-val" {l_col}>{s['L']}</div>
        <div class="pt-val">{nrr_str}</div>
        <div class="pt-pts">{s['PTS']}</div>
      </div>""")
    return '\n'.join(rows)

print('✅ Render helpers ready')

In [ ]:
# ─── FULL HTML STRING ─────────────────────────────────────────────────────────
updated_at = datetime.now().strftime('%B %d, %Y at %H:%M')

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>IPL {TARGET_SEASON} — Season Analytics</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Bebas+Neue&family=Syne:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg:#070710;--surface:#0f0f1c;--card:#14142a;--card2:#1c1c36;
    --border:#252540;--border2:#32325a;
    --gold:#f0b429;--gold2:#ff8c00;--red:#ff4560;--blue:#00d4ff;
    --green:#00e5a0;--purple:#a855f7;--text:#eeeef8;--muted:#7070a0;--dim:#3a3a60;
  }}
  *{{margin:0;padding:0;box-sizing:border-box;}}
  html{{scroll-behavior:smooth;}}
  body{{background:var(--bg);color:var(--text);font-family:'Syne',sans-serif;overflow-x:hidden;}}
  body::before{{content:'';position:fixed;inset:0;
    background-image:linear-gradient(rgba(240,180,41,0.03) 1px,transparent 1px),
      linear-gradient(90deg,rgba(240,180,41,0.03) 1px,transparent 1px);
    background-size:60px 60px;pointer-events:none;z-index:0;}}
  .blob1,.blob2,.blob3{{position:fixed;border-radius:50%;filter:blur(120px);pointer-events:none;z-index:0;opacity:0.12;}}
  .blob1{{width:600px;height:600px;background:var(--gold);top:-200px;right:-200px;}}
  .blob2{{width:500px;height:500px;background:var(--red);bottom:200px;left:-200px;}}
  .blob3{{width:400px;height:400px;background:var(--blue);top:50%;right:10%;}}
  nav{{position:sticky;top:0;z-index:100;background:rgba(7,7,16,0.9);backdrop-filter:blur(20px);
    border-bottom:1px solid var(--border);padding:0 40px;display:flex;align-items:center;
    justify-content:space-between;height:56px;}}
  .nav-brand{{font-family:'Bebas Neue',sans-serif;font-size:22px;letter-spacing:0.1em;color:var(--gold);}}
  .nav-links{{display:flex;gap:32px;list-style:none;}}
  .nav-links a{{font-size:12px;letter-spacing:2px;text-transform:uppercase;color:var(--muted);
    text-decoration:none;font-family:'JetBrains Mono',monospace;transition:color 0.2s;}}
  .nav-links a:hover{{color:var(--gold);}}
  .nav-live{{display:flex;align-items:center;gap:8px;font-size:11px;
    font-family:'JetBrains Mono',monospace;color:var(--green);letter-spacing:1px;}}
  .live-dot{{width:7px;height:7px;background:var(--green);border-radius:50%;animation:pulse 2s infinite;}}
  @keyframes pulse{{0%,100%{{opacity:1;transform:scale(1);}}50%{{opacity:0.4;transform:scale(0.7);}}}}
  .hero{{position:relative;z-index:1;min-height:90vh;display:flex;flex-direction:column;
    justify-content:center;align-items:center;text-align:center;padding:80px 40px;overflow:hidden;}}
  .hero-badge{{display:inline-flex;align-items:center;gap:10px;padding:8px 20px;
    border:1px solid rgba(240,180,41,0.3);border-radius:30px;background:rgba(240,180,41,0.06);
    font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:2px;
    text-transform:uppercase;color:var(--gold);margin-bottom:32px;
    opacity:0;animation:slideDown 0.6s 0.2s forwards;}}
  .hero-title{{font-family:'Bebas Neue',sans-serif;font-size:clamp(70px,14vw,160px);
    line-height:0.88;letter-spacing:0.02em;margin-bottom:28px;
    opacity:0;animation:slideDown 0.7s 0.35s forwards;}}
  .hero-title .y{{color:var(--gold);}}
  .hero-title .line2{{display:block;font-size:clamp(40px,7vw,80px);color:var(--muted);}}
  .hero-desc{{font-size:16px;color:var(--muted);max-width:560px;line-height:1.7;margin-bottom:60px;
    opacity:0;animation:slideDown 0.7s 0.5s forwards;}}
  .hero-desc strong{{color:var(--text);}}
  .hero-stats{{display:flex;gap:0;border:1px solid var(--border);border-radius:16px;overflow:hidden;
    background:rgba(20,20,42,0.8);backdrop-filter:blur(10px);
    opacity:0;animation:slideDown 0.7s 0.65s forwards;}}
  .hs-item{{padding:28px 40px;border-right:1px solid var(--border);text-align:center;}}
  .hs-item:last-child{{border-right:none;}}
  .hs-num{{font-family:'Bebas Neue',sans-serif;font-size:48px;line-height:1;color:var(--gold);display:block;}}
  .hs-label{{font-family:'JetBrains Mono',monospace;font-size:10px;letter-spacing:2px;
    text-transform:uppercase;color:var(--muted);margin-top:6px;}}
  .section{{position:relative;z-index:1;padding:90px 0;}}
  .container{{max-width:1160px;margin:0 auto;padding:0 40px;}}
  .section-tag{{font-family:'JetBrains Mono',monospace;font-size:10px;letter-spacing:3px;
    text-transform:uppercase;color:var(--gold);margin-bottom:10px;}}
  .section-title{{font-family:'Bebas Neue',sans-serif;font-size:58px;letter-spacing:0.04em;
    line-height:1;margin-bottom:10px;}}
  .section-sub{{font-size:15px;color:var(--muted);margin-bottom:52px;max-width:580px;}}
  .divider{{height:1px;background:linear-gradient(90deg,transparent,var(--border2),transparent);margin:0 40px;}}
  .toss-grid{{display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:28px;}}
  .toss-card{{padding:36px;border-radius:16px;border:1px solid var(--border2);background:var(--card);
    position:relative;overflow:hidden;}}
  .toss-card::before{{content:'';position:absolute;inset:0;
    background:radial-gradient(ellipse at top left,var(--glow-c) 0%,transparent 70%);
    opacity:0.07;pointer-events:none;}}
  .toss-card.win{{--glow-c:var(--green);}}.toss-card.lose{{--glow-c:var(--red);}}
  .tc-eyebrow{{font-family:'JetBrains Mono',monospace;font-size:10px;letter-spacing:2px;
    text-transform:uppercase;color:var(--muted);margin-bottom:12px;}}
  .tc-pct{{font-family:'Bebas Neue',sans-serif;font-size:88px;line-height:1;margin-bottom:6px;}}
  .tc-pct.win{{color:var(--green);}}.tc-pct.lose{{color:var(--red);}}
  .tc-label{{font-size:14px;color:var(--muted);margin-bottom:24px;}}
  .tc-bar-track{{height:6px;background:var(--border);border-radius:3px;overflow:hidden;margin-bottom:16px;}}
  .tc-bar-fill{{height:100%;border-radius:3px;width:0;transition:width 1.6s cubic-bezier(0.16,1,0.3,1);}}
  .tc-bar-fill.win{{background:linear-gradient(90deg,var(--green),#00ff88);}}
  .tc-bar-fill.lose{{background:linear-gradient(90deg,var(--red),#ff8080);}}
  .tc-stats{{display:flex;gap:20px;}}
  .tc-stat-item{{background:var(--surface);border:1px solid var(--border);border-radius:8px;
    padding:10px 14px;flex:1;}}
  .tc-stat-n{{font-family:'Bebas Neue',sans-serif;font-size:24px;color:var(--text);display:block;line-height:1.1;}}
  .tc-stat-l{{font-size:10px;color:var(--muted);font-family:'JetBrains Mono',monospace;letter-spacing:1px;}}
  .toss-breakdown{{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:24px;}}
  .tb-item{{background:var(--card);border:1px solid var(--border);border-radius:12px;padding:20px;text-align:center;}}
  .tb-n{{font-family:'Bebas Neue',sans-serif;font-size:36px;color:var(--gold);display:block;line-height:1;}}
  .tb-l{{font-size:11px;color:var(--muted);font-family:'JetBrains Mono',monospace;letter-spacing:1px;margin-top:4px;}}
  .insight-box{{background:var(--card);border:1px solid var(--border2);border-left:3px solid var(--gold);
    border-radius:12px;padding:20px 24px;display:flex;gap:16px;align-items:flex-start;}}
  .ib-icon{{font-size:22px;flex-shrink:0;margin-top:2px;}}
  .ib-text{{font-size:14px;line-height:1.7;color:var(--muted);}}
  .ib-text strong{{color:var(--text);}}
  .phase-wrap{{display:flex;flex-direction:column;gap:16px;}}
  .phase-block{{background:var(--card);border:1px solid var(--border);border-radius:14px;
    padding:24px 28px;display:grid;grid-template-columns:160px 1fr;gap:32px;
    align-items:center;transition:border-color 0.2s;}}
  .phase-block:hover{{border-color:var(--border2);}}
  .phase-block.highlight{{border-color:var(--gold);background:rgba(240,180,41,0.04);}}
  .pb-name{{font-family:'Bebas Neue',sans-serif;font-size:26px;letter-spacing:0.04em;line-height:1;}}
  .pb-overs{{font-family:'JetBrains Mono',monospace;font-size:11px;color:var(--muted);letter-spacing:1px;margin-top:4px;}}
  .pb-gap-badge{{display:inline-block;padding:4px 10px;border-radius:20px;
    font-family:'JetBrains Mono',monospace;font-size:12px;font-weight:500;margin-top:10px;
    background:rgba(240,180,41,0.12);color:var(--gold);border:1px solid rgba(240,180,41,0.25);}}
  .pb-bar-row{{display:flex;flex-direction:column;gap:10px;}}
  .pb-bar-label{{display:flex;justify-content:space-between;font-size:11px;
    font-family:'JetBrains Mono',monospace;color:var(--muted);letter-spacing:1px;
    text-transform:uppercase;margin-bottom:2px;}}
  .pb-bar-track{{height:22px;background:var(--border);border-radius:6px;overflow:hidden;position:relative;}}
  .pb-bar-fill{{height:100%;border-radius:6px;position:relative;display:flex;align-items:center;
    padding-right:10px;justify-content:flex-end;width:0;
    transition:width 1.5s cubic-bezier(0.16,1,0.3,1);
    font-family:'JetBrains Mono',monospace;font-size:11px;font-weight:500;white-space:nowrap;}}
  .pb-bar-fill.winner{{background:linear-gradient(90deg,rgba(0,229,160,0.3),var(--green));color:var(--bg);}}
  .pb-bar-fill.loser{{background:linear-gradient(90deg,rgba(255,69,96,0.3),var(--red));color:var(--bg);}}
  .phase-legend{{display:flex;gap:24px;margin-bottom:20px;}}
  .pl-item{{display:flex;align-items:center;gap:8px;font-size:12px;
    font-family:'JetBrains Mono',monospace;color:var(--muted);letter-spacing:1px;}}
  .pl-dot{{width:10px;height:10px;border-radius:3px;}}
  .tables-row{{display:grid;grid-template-columns:1fr 1fr;gap:24px;}}
  .table-card{{background:var(--card);border:1px solid var(--border);border-radius:16px;overflow:hidden;}}
  .table-header{{padding:22px 28px 18px;border-bottom:1px solid var(--border);
    display:flex;align-items:center;justify-content:space-between;background:var(--card2);}}
  .th-title{{font-family:'Bebas Neue',sans-serif;font-size:26px;letter-spacing:0.05em;}}
  .th-cap{{font-family:'JetBrains Mono',monospace;font-size:10px;letter-spacing:2px;
    padding:5px 12px;border-radius:20px;text-transform:uppercase;}}
  .th-cap.orange{{background:rgba(255,140,0,0.15);color:var(--gold2);border:1px solid rgba(255,140,0,0.3);}}
  .th-cap.purple{{background:rgba(168,85,247,0.15);color:var(--purple);border:1px solid rgba(168,85,247,0.3);}}
  .player-row{{display:grid;grid-template-columns:44px 1fr auto;align-items:center;gap:14px;
    padding:16px 28px;border-bottom:1px solid var(--border);transition:background 0.15s;cursor:default;}}
  .player-row:last-child{{border-bottom:none;}}
  .player-row:hover{{background:rgba(255,255,255,0.02);}}
  .pr-rank{{font-family:'Bebas Neue',sans-serif;font-size:28px;line-height:1;text-align:center;}}
  .pr-rank.r1{{color:#f0b429;}}.pr-rank.r2{{color:#c0c0d0;}}.pr-rank.r3{{color:#cd7f32;}}.pr-rank.rn{{color:var(--dim);}}
  .pr-name{{font-size:15px;font-weight:600;color:var(--text);line-height:1.2;}}
  .pr-meta{{font-family:'JetBrains Mono',monospace;font-size:10px;color:var(--muted);letter-spacing:1px;margin-top:2px;}}
  .pr-stats{{text-align:right;}}
  .pr-main{{font-family:'Bebas Neue',sans-serif;font-size:30px;line-height:1;display:block;}}
  .pr-main.bat{{color:var(--gold2);}}.pr-main.bowl{{color:var(--purple);}}
  .pr-sub{{font-family:'JetBrains Mono',monospace;font-size:10px;color:var(--muted);letter-spacing:1px;}}
  .surprise-section{{position:relative;z-index:1;padding:90px 0 100px;}}
  .surprise-inner{{background:var(--card);border:1px solid var(--border2);border-radius:24px;
    padding:64px;text-align:center;position:relative;overflow:hidden;}}
  .surprise-inner::after{{content:'';position:absolute;inset:0;
    background:radial-gradient(ellipse at 50% 0%,rgba(240,180,41,0.07) 0%,transparent 65%);
    pointer-events:none;}}
  .surprise-tag{{position:relative;z-index:1;font-family:'JetBrains Mono',monospace;
    font-size:10px;letter-spacing:3px;text-transform:uppercase;color:var(--gold);margin-bottom:24px;}}
  .surprise-headline{{position:relative;z-index:1;font-family:'Bebas Neue',sans-serif;
    font-size:clamp(30px,4vw,52px);line-height:1.15;letter-spacing:0.03em;
    max-width:820px;margin:0 auto 24px;}}
  .surprise-headline .accent{{color:var(--gold);}}
  .surprise-headline .danger{{color:var(--red);}}
  .surprise-headline .safe{{color:var(--green);}}
  .surprise-detail{{position:relative;z-index:1;font-size:15px;color:var(--muted);
    max-width:660px;margin:0 auto 48px;line-height:1.8;}}
  .surprise-detail strong{{color:var(--text);}}
  .answers-row{{display:grid;grid-template-columns:repeat(3,1fr);gap:20px;position:relative;z-index:1;}}
  .ans-card{{background:var(--surface);border:1px solid var(--border);border-radius:12px;padding:24px;text-align:left;}}
  .ans-num{{font-family:'Bebas Neue',sans-serif;font-size:48px;line-height:1;margin-bottom:8px;}}
  .ans-num.g{{color:var(--green);}}.ans-num.r{{color:var(--red);}}.ans-num.b{{color:var(--blue);}}
  .ans-label{{font-family:'JetBrains Mono',monospace;font-size:10px;color:var(--muted);
    letter-spacing:2px;text-transform:uppercase;margin-bottom:10px;}}
  .ans-text{{font-size:13px;color:var(--muted);line-height:1.6;}}
  .ans-text strong{{color:var(--text);}}
  .points-table{{background:var(--card);border:1px solid var(--border);border-radius:16px;
    overflow:hidden;margin-bottom:40px;}}
  .pt-header{{display:grid;grid-template-columns:30px 1fr repeat(4,60px);gap:10px;
    padding:14px 24px;background:var(--card2);border-bottom:1px solid var(--border);
    font-family:'JetBrains Mono',monospace;font-size:10px;letter-spacing:2px;
    text-transform:uppercase;color:var(--muted);}}
  .pt-row{{display:grid;grid-template-columns:30px 1fr repeat(4,60px);gap:10px;
    padding:12px 24px;border-bottom:1px solid var(--border);align-items:center;transition:background 0.15s;}}
  .pt-row:last-child{{border-bottom:none;}}
  .pt-row:hover{{background:rgba(255,255,255,0.02);}}
  .pt-row.qualified{{border-left:3px solid var(--green);}}
  .pt-row.eliminated{{border-left:3px solid var(--red);opacity:0.65;}}
  .pt-pos{{font-family:'JetBrains Mono',monospace;font-size:12px;color:var(--muted);}}
  .pt-team{{font-size:14px;font-weight:600;}}
  .pt-team-code{{font-family:'JetBrains Mono',monospace;font-size:10px;color:var(--muted);}}
  .pt-val{{font-family:'JetBrains Mono',monospace;font-size:13px;text-align:right;}}
  .pt-pts{{font-family:'Bebas Neue',sans-serif;font-size:20px;color:var(--gold);text-align:right;}}
  footer{{position:relative;z-index:1;border-top:1px solid var(--border);
    padding:32px 40px;display:flex;align-items:center;justify-content:space-between;
    font-family:'JetBrains Mono',monospace;font-size:11px;color:var(--muted);letter-spacing:1px;}}
  .footer-brand{{font-family:'Bebas Neue',sans-serif;font-size:18px;color:var(--gold);letter-spacing:0.1em;}}
  @keyframes slideDown{{from{{opacity:0;transform:translateY(-16px);}}to{{opacity:1;transform:translateY(0);}}}}
  .reveal{{opacity:0;transform:translateY(28px);transition:opacity 0.7s ease,transform 0.7s ease;}}
  .reveal.in{{opacity:1;transform:translateY(0);}}
  @media(max-width:768px){{
    nav{{padding:0 16px;}}.nav-links{{display:none;}}.container{{padding:0 16px;}}
    .hero{{padding:60px 16px;min-height:auto;}}
    .hero-stats{{flex-direction:column;width:100%;}}
    .hs-item{{border-right:none;border-bottom:1px solid var(--border);}}
    .toss-grid{{grid-template-columns:1fr;}}.toss-breakdown{{grid-template-columns:1fr 1fr;}}
    .phase-block{{grid-template-columns:1fr;}}.tables-row{{grid-template-columns:1fr;}}
    .answers-row{{grid-template-columns:1fr;}}.surprise-inner{{padding:40px 24px;}}
    footer{{flex-direction:column;gap:10px;text-align:center;}}
  }}
</style>
</head>
<body>
<div class="blob1"></div><div class="blob2"></div><div class="blob3"></div>

<nav>
  <div class="nav-brand">IPL {TARGET_SEASON}</div>
  <ul class="nav-links">
    <li><a href="#toss">Toss</a></li>
    <li><a href="#phases">Phases</a></li>
    <li><a href="#players">Players</a></li>
    <li><a href="#standings">Standings</a></li>
    <li><a href="#insight">Finding</a></li>
  </ul>
  <div class="nav-live"><div class="live-dot"></div>DATA LIVE · {TARGET_SEASON}</div>
</nav>

<section class="hero">
  <div class="hero-badge"><div class="live-dot"></div>IPL {TARGET_SEASON} Season · Ball-by-Ball Data · {hero['total_matches']} Matches</div>
  <h1 class="hero-title">IPL <span class="y">{TARGET_SEASON}</span><span class="line2">Your Opinions, Numbered</span></h1>
  <p class="hero-desc">Three questions. Actual {TARGET_SEASON} season data. <strong>No guesses.</strong><br>We ran the ball-by-ball numbers so you don't have to.</p>
  <div class="hero-stats">
    <div class="hs-item"><span class="hs-num">{hero['total_matches']}</span><div class="hs-label">League Matches</div></div>
    <div class="hs-item"><span class="hs-num">{hero['total_teams']}</span><div class="hs-label">Teams</div></div>
    <div class="hs-item"><span class="hs-num">{hero['top_bat_runs']}</span><div class="hs-label">Top Runs ({hero['top_bat_name'].split()[-1]})</div></div>
    <div class="hs-item"><span class="hs-num">{hero['top_bowl_wkts']}</span><div class="hs-label">Top Wkts ({hero['top_bowl_name'].split()[-1]})</div></div>
  </div>
</section>

<div class="divider"></div>

<section class="section reveal" id="toss">
  <div class="container">
    <div class="section-tag">01 / Question One</div>
    <h2 class="section-title">THE TOSS TRUTH</h2>
    <p class="section-sub">Do teams that win the toss actually win more matches in IPL {TARGET_SEASON}?</p>
    <div class="toss-grid">
      <div class="toss-card win">
        <div class="tc-eyebrow">Toss Winners — Win Rate</div>
        <div class="tc-pct win">{toss['toss_win_pct']}%</div>
        <div class="tc-label">{toss['toss_wins']} wins out of {toss['total_matches']} league matches</div>
        <div class="tc-bar-track"><div class="tc-bar-fill win" data-w="{toss['toss_win_pct']}"></div></div>
        <div class="tc-stats">
          <div class="tc-stat-item"><span class="tc-stat-n">{toss['toss_wins']}</span><div class="tc-stat-l">MATCHES WON</div></div>
          <div class="tc-stat-item"><span class="tc-stat-n">{toss['toss_losses']}</span><div class="tc-stat-l">MATCHES LOST</div></div>
        </div>
      </div>
      <div class="toss-card lose">
        <div class="tc-eyebrow">Toss Losers — Win Rate</div>
        <div class="tc-pct lose">{toss['toss_lose_pct']}%</div>
        <div class="tc-label">{toss['toss_losses']} wins out of {toss['total_matches']} league matches</div>
        <div class="tc-bar-track"><div class="tc-bar-fill lose" data-w="{toss['toss_lose_pct']}"></div></div>
        <div class="tc-stats">
          <div class="tc-stat-item"><span class="tc-stat-n">{toss['toss_losses']}</span><div class="tc-stat-l">MATCHES WON</div></div>
          <div class="tc-stat-item"><span class="tc-stat-n">{toss['toss_wins']}</span><div class="tc-stat-l">MATCHES LOST</div></div>
        </div>
      </div>
    </div>
    <div class="toss-breakdown">
      <div class="tb-item"><span class="tb-n">{toss['field_pct']}%</span><div class="tb-l">Chose to field first</div></div>
      <div class="tb-item"><span class="tb-n">{toss['field_win_pct']}%</span><div class="tb-l">Field-first win rate</div></div>
      <div class="tb-item"><span class="tb-n">{toss['bat_win_pct']}%</span><div class="tb-l">Bat-first win rate</div></div>
      <div class="tb-item"><span class="tb-n">{toss['margin_pct']}%</span><div class="tb-l">Margin above coin flip</div></div>
    </div>
    <div class="insight-box">
      <div class="ib-icon">🎯</div>
      <p class="ib-text">Toss winners win <strong>{toss['toss_win_pct']}%</strong> of the time — a real but modest edge over 50/50.
      <strong>Fielding first ({toss['field_win_pct']}% win rate)</strong> is overwhelmingly the preferred toss call in {TARGET_SEASON}.
      Teams choosing to bat first win just {toss['bat_win_pct']}% of those games.</p>
    </div>
  </div>
</section>

<div class="divider"></div>

<section class="section reveal" id="phases">
  <div class="container">
    <div class="section-tag">02 / Question Two</div>
    <h2 class="section-title">PHASE BREAKDOWN</h2>
    <p class="section-sub">Which phase of innings separates winners from losers most in IPL {TARGET_SEASON}?</p>
    <div class="phase-legend">
      <div class="pl-item"><div class="pl-dot" style="background:var(--green)"></div>WINNER AVG</div>
      <div class="pl-item"><div class="pl-dot" style="background:var(--red)"></div>LOSER AVG</div>
    </div>
    <div class="phase-wrap">
{render_phase_blocks(phases_data)}
    </div>
    <div class="insight-box" style="margin-top:28px">
      <div class="ib-icon">📊</div>
      <p class="ib-text">The <strong>{biggest_gap_phase} overs</strong> show the biggest gap between winners and losers.
      Teams dominating the {biggest_gap_phase.lower()} phase are far more likely to win the match.</p>
    </div>
  </div>
</section>

<div class="divider"></div>

<section class="section reveal" id="players">
  <div class="container">
    <div class="section-tag">03 / Question Three</div>
    <h2 class="section-title">THE CHARTS</h2>
    <p class="section-sub">Orange Cap &amp; Purple Cap standings — scraped live from ball-by-ball data.</p>
    <div class="tables-row">
      <div class="table-card">
        <div class="table-header">
          <div class="th-title">ORANGE CAP</div>
          <div class="th-cap orange">TOP RUNS</div>
        </div>
{render_batter_rows(top_batters)}
      </div>
      <div class="table-card">
        <div class="table-header">
          <div class="th-title">PURPLE CAP</div>
          <div class="th-cap purple">TOP WICKETS</div>
        </div>
{render_bowler_rows(top_bowlers)}
      </div>
    </div>
  </div>
</section>

<div class="divider"></div>

<section class="section reveal" id="standings">
  <div class="container">
    <div class="section-tag">04 / Bonus — Points Table</div>
    <h2 class="section-title">{TARGET_SEASON} STANDINGS</h2>
    <p class="section-sub">League stage standings computed directly from ball-by-ball CSV data.</p>
    <div class="points-table">
      <div class="pt-header">
        <div>#</div><div>Team</div>
        <div style="text-align:right">W</div>
        <div style="text-align:right">L</div>
        <div style="text-align:right">NRR</div>
        <div style="text-align:right">PTS</div>
      </div>
{render_standings_rows(standings)}
    </div>
    <div class="insight-box">
      <div class="ib-icon">🏆</div>
      <p class="ib-text">Top 4 teams qualify for playoffs. <strong>{standings[0]['team']}</strong> leads the table with {standings[0]['PTS']} points and NRR {sign(standings[0]['NRR'])}.</p>
    </div>
  </div>
</section>

<div class="divider"></div>

<section class="surprise-section reveal" id="insight">
  <div class="container">
    <div class="section-tag">05 / The Finding</div>
    <h2 class="section-title">WHAT THE DATA SAYS</h2>
    <div class="surprise-inner">
      <div class="surprise-tag">⚡ Key insights from {TARGET_SEASON}</div>
      <div class="surprise-headline">
        <span class="accent">{surprise['top_bat_name']}</span> leads with
        <span class="accent">{surprise['top_bat_runs']} runs</span> —
        while <span class="safe">{surprise['top_bowl_name']}</span> tops the wicket charts with
        <span class="accent">{surprise['top_bowl_wkts']} wickets.</span>
      </div>
      <p class="surprise-detail">
        Field-first strategy dominates with a <strong>{surprise['field_win_pct']}% win rate</strong>.
        The death overs create the biggest separation between winners and losers —
        a <strong>+{surprise['death_gap']} run gap</strong> between winning and losing innings.
        Toss advantage is real but marginal at {surprise['toss_win_pct']}%.
      </p>
      <div class="answers-row">
        <div class="ans-card">
          <div class="ans-num g">{surprise['toss_win_pct']}%</div>
          <div class="ans-label">Toss winners' win rate</div>
          <div class="ans-text">A real but small edge. <strong>Fielding first ({surprise['field_win_pct']}% win rate)</strong> is the smart toss call in {TARGET_SEASON}.</div>
        </div>
        <div class="ans-card">
          <div class="ans-num r">+{surprise['death_gap']}</div>
          <div class="ans-label">Death overs gap (runs)</div>
          <div class="ans-text"><strong>Death overs decide matches</strong> — the biggest phase gap between winning and losing teams.</div>
        </div>
        <div class="ans-card">
          <div class="ans-num b">{surprise['top_bowl_wkts']}</div>
          <div class="ans-label">Purple Cap leader wickets</div>
          <div class="ans-text"><strong>{surprise['top_bowl_name']}</strong> leads the Purple Cap. <strong>{surprise['top_bat_name']}</strong> leads run charts. {TARGET_SEASON} data speaks.</div>
        </div>
      </div>
    </div>
  </div>
</section>

<footer>
  <div class="footer-brand">IPL {TARGET_SEASON}</div>
  <div>DATA: CRICSHEET.ORG · AUTO-GENERATED FROM CSV · UPDATED {updated_at.upper()}</div>
  <div>BCCI / IPL · BALL-BY-BALL ANALYSIS</div>
</footer>

<script>
  const io = new IntersectionObserver((entries) => {{
    entries.forEach(entry => {{
      if (!entry.isIntersecting) return;
      entry.target.classList.add('in');
      entry.target.querySelectorAll('.tc-bar-fill[data-w]').forEach(el => {{
        setTimeout(() => {{ el.style.width = el.dataset.w + '%'; }}, 200);
      }});
      entry.target.querySelectorAll('.pb-bar-fill[data-w]').forEach(el => {{
        setTimeout(() => {{ el.style.width = el.dataset.w + '%'; }}, 300);
      }});
      io.unobserve(entry.target);
    }});
  }}, {{ threshold: 0.12 }});
  document.querySelectorAll('.reveal').forEach(el => io.observe(el));
  const playerIo = new IntersectionObserver((entries) => {{
    entries.forEach(entry => {{
      if (!entry.isIntersecting) return;
      entry.target.querySelectorAll('.player-row').forEach((row, i) => {{
        row.style.opacity = '0'; row.style.transform = 'translateX(-12px)';
        setTimeout(() => {{
          row.style.transition = 'opacity 0.4s ease, transform 0.4s ease';
          row.style.opacity = '1'; row.style.transform = 'translateX(0)';
        }}, i * 80 + 200);
      }});
      playerIo.unobserve(entry.target);
    }});
  }}, {{ threshold: 0.1 }});
  document.querySelectorAll('.table-card').forEach(el => playerIo.observe(el));
</script>
</body>
</html>"""

print(f'✅ HTML generated ({len(html):,} characters)')

## Cell 10 — Save HTML

In [ ]:
# ─── SAVE TO FILE ────────────────────────────────────────────────────────────
OUTPUT_FILE = f'ipl_{TARGET_SEASON}_analytics.html'

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(html)

size_kb = round(os.path.getsize(OUTPUT_FILE) / 1024, 1)
print(f'✅ Saved: {OUTPUT_FILE}  ({size_kb} KB)')
print(f'   Open in browser: file://{os.path.abspath(OUTPUT_FILE)}')
print()
print('─' * 60)
print('💡 To update: change the CSV, re-run all cells (Kernel > Restart & Run All)')
print('   The HTML will regenerate with fresh stats automatically.')